# Semantic-Adaptive 3DGS：T4 全链路测试

这个 Notebook 使用 **NeRF Synthetic Lego** 标准多视角数据集的 8 个稀疏训练视角，实际编译并运行官方 CUDA Gaussian Rasterizer，然后验证：SAM+CLIP 预处理 → 重要区域加权 RGB 3DGS → 语义蒸馏 → 文本搜索 → 非破坏性删除。重要区域词汇已由项目助手预生成，不调用外部 LLM API。

运行前在 `Runtime / 运行时 → Change runtime type / 更改运行时类型` 中选择 **T4 GPU**。不要选择 A100/H100。本测试是功能冒烟测试，300 次 RGB 迭代不代表最终重建质量。

In [ ]:
import os, platform, subprocess, sys, time
import torch

assert torch.cuda.is_available(), '请先把 Colab 运行时改为 T4 GPU，然后重新运行。'
GPU_NAME = torch.cuda.get_device_name(0)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', GPU_NAME)
if 'T4' not in GPU_NAME.upper():
    raise RuntimeError(f'当前是 {GPU_NAME}。为避免超预算，请断开后明确选择 T4 GPU。')
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Xuyw041006-arch/gaussian-splatting.git'
REPO = Path('/content/gaussian-splatting')
RUN_ROOT = Path('/content/semantic_3dgs_t4_smoke')
SCENE = RUN_ROOT / 'lego_sparse8'
MODEL = RUN_ROOT / 'model'
EDITED_MODEL = RUN_ROOT / 'model_without_query'
SAM_CHECKPOINT = Path('/content/checkpoints/sam_vit_b_01ec64.pth')
TRAIN_VIEWS = 8
TEST_VIEWS = 2
IMAGE_SIZE = 256
INITIAL_POINTS = 8000
RGB_ITERATIONS = 300
SEMANTIC_ITERATIONS = 60
IMPORTANT_OBJECTS = [
    'yellow Lego bulldozer',
    'yellow vehicle body',
    'black excavator arm',
    'gray caterpillar track',
    'red wheel hub',
]
print('测试目录:', RUN_ROOT)


## 1. 克隆代码并编译真实 CUDA 扩展

第一次运行通常最慢，因为这里会现场编译官方 rasterizer；这一步成功才算真正具备 3DGS 运行环境。

In [ ]:
def run(command, cwd=None):
    printable = ' '.join(map(str, command))
    print(f'\n$ {printable}')
    started = time.time()
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)
    print(f'完成，用时 {(time.time() - started) / 60:.1f} 分钟')

if not REPO.exists():
    run(['git', 'clone', '--recursive', REPO_URL, REPO])
else:
    run(['git', '-C', REPO, 'pull', '--ff-only'])
    run(['git', '-C', REPO, 'submodule', 'update', '--init', '--recursive'])

os.environ['MAX_JOBS'] = '2'
run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'plyfile', 'huggingface_hub'])
for package in ('diff-gaussian-rasterization', 'simple-knn', 'fused-ssim'):
    run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', REPO / 'submodules' / package])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO / 'requirements-semantic.txt'])
run([sys.executable, REPO / 'scripts/preflight.py'], cwd=REPO)


## 2. 下载标准 Lego 数据并构造 8 视角稀疏场景

只下载用到的 8 张训练图和 2 张测试图，随后缩放到 256 像素，减少 T4 时间和学生额度消耗。数据来自公开的 NeRF Synthetic 镜像。

In [ ]:
import json, shutil
import numpy as np
from huggingface_hub import hf_hub_download

DATA_REPO = 'phuckstnk63/nerf-synthetic'
RAW_ROOT = Path('/content/nerf_synthetic_raw')
FULL_SCENE = Path('/content/nerf_synthetic_stage/lego')
if FULL_SCENE.exists():
    shutil.rmtree(FULL_SCENE)
(FULL_SCENE / 'train').mkdir(parents=True, exist_ok=True)
json_name = 'nerf_synthetic/lego/transforms_train.json'
json_path = Path(hf_hub_download(DATA_REPO, json_name, repo_type='dataset', local_dir=RAW_ROOT))
transforms = json.loads(json_path.read_text())
indices = np.linspace(0, len(transforms['frames']) - 1, TRAIN_VIEWS + TEST_VIEWS, dtype=int)
held_out_positions = set(np.linspace(1, len(indices) - 2, TEST_VIEWS, dtype=int).tolist())
train_frames, test_frames = [], []
for position, index in enumerate(indices):
    frame = transforms['frames'][int(index)]
    relative = frame['file_path'].removeprefix('./') + '.png'
    downloaded = Path(hf_hub_download(DATA_REPO, f'nerf_synthetic/lego/{relative}', repo_type='dataset', local_dir=RAW_ROOT))
    destination = FULL_SCENE / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(downloaded, destination)
    (test_frames if position in held_out_positions else train_frames).append(frame)
for split, frames in (('train', train_frames), ('test', test_frames)):
    payload = {'camera_angle_x': transforms['camera_angle_x'], 'frames': frames}
    (FULL_SCENE / f'transforms_{split}.json').write_text(json.dumps(payload, indent=2))
assert len(train_frames) == TRAIN_VIEWS and len(test_frames) == TEST_VIEWS

if SCENE.exists():
    shutil.rmtree(SCENE)
run([sys.executable, REPO / 'scripts/prepare_nerf_sparse_scene.py',
     '--source', FULL_SCENE, '--output', SCENE, '--train_views', str(TRAIN_VIEWS),
     '--test_views', str(TEST_VIEWS), '--size', str(IMAGE_SIZE),
     '--points', str(INITIAL_POINTS)], cwd=REPO)

from PIL import Image
from IPython.display import display
display(Image.open(sorted((SCENE / 'images').glob('*.png'))[0]).resize((256, 256)))
IMPORTANT_JSON = SCENE / 'important_objects.generated.json'
prompt_map = {path.name: IMPORTANT_OBJECTS for path in sorted((SCENE / 'images').glob('*.png'))}
IMPORTANT_JSON.write_text(json.dumps(prompt_map, ensure_ascii=False, indent=2))
print('助手预生成的重要区域词汇:', json.dumps(IMPORTANT_OBJECTS, ensure_ascii=False))


## 3. SAM + OpenCLIP 语义与重要区域

这里下载较小的 SAM ViT-B 权重，并读取上一单元生成的 `important_objects.generated.json`。它与将来 LLM API 的输出格式一致，但本次不产生 API 调用或费用。

In [ ]:
import urllib.request

SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not SAM_CHECKPOINT.exists():
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        SAM_CHECKPOINT,
    )
run([sys.executable, REPO / 'preprocess_semantics.py',
     '--scene', SCENE, '--sam_checkpoint', SAM_CHECKPOINT, '--sam_model', 'vit_b',
     '--clip_model', 'ViT-B-16', '--clip_pretrained', 'laion2b_s34b_b88k',
     '--important_json', IMPORTANT_JSON,
     '--feature_dim', '6', '--feature_width', '128', '--min_mask_area', '64',
     '--max_masks', '32', '--points_per_side', '12', '--batch_size', '16',
     '--importance_topk', '2', '--device', 'cuda'], cwd=REPO)
assert (SCENE / 'semantic_meta.npz').is_file()
assert len(list((SCENE / 'semantic_maps').glob('*.npz'))) == TRAIN_VIEWS
print((SCENE / 'semantic_summary.json').read_text())


## 4. 重要区域加权 RGB 3DGS（真实 CUDA 训练）

In [ ]:
if MODEL.exists():
    shutil.rmtree(MODEL)
run([sys.executable, REPO / 'train.py', '-s', SCENE, '-m', MODEL, '--eval',
     '--white_background', '--iterations', str(RGB_ITERATIONS),
     '--save_iterations', str(RGB_ITERATIONS), '--test_iterations', str(RGB_ITERATIONS),
     '--importance_mask_dir', SCENE / 'importance_masks',
     '--foreground_weight', '4.0', '--background_weight', '0.25',
     '--densify_from_iter', '50', '--densify_until_iter', '250',
     '--densification_interval', '50', '--opacity_reset_interval', '1000',
     '--disable_viewer', '--quiet'], cwd=REPO)
RGB_PLY = MODEL / 'point_cloud' / f'iteration_{RGB_ITERATIONS}' / 'point_cloud.ply'
assert RGB_PLY.is_file(), RGB_PLY
print('RGB 3DGS 输出:', RGB_PLY, RGB_PLY.stat().st_size, 'bytes')


## 5. 把二维语义蒸馏到固定的三维高斯

In [ ]:
run([sys.executable, REPO / 'train_semantics.py', '-m', MODEL,
     '--iteration', str(RGB_ITERATIONS),
     '--semantic_iterations', str(SEMANTIC_ITERATIONS),
     '--semantic_lr', '0.01', '--save_every', '0', '--quiet'], cwd=REPO)
SEMANTIC_FILE = MODEL / 'semantic' / f'iteration_{RGB_ITERATIONS}' / 'semantic_features.pt'
assert SEMANTIC_FILE.is_file(), SEMANTIC_FILE
print('三维语义输出:', SEMANTIC_FILE, SEMANTIC_FILE.stat().st_size, 'bytes')


## 6. 文本搜索和删除验证

300/60 次迭代只验证功能，所以查询用宽松阈值并保留分数最高的 500 个高斯。正式训练后应改用验证集选择阈值。删除会写入新模型，原模型保持不变。

In [ ]:
SELECTION = RUN_ROOT / 'yellow_lego_selection.npz'
QUERY_JSON = RUN_ROOT / 'yellow_lego_result.json'
SELECTED_PLY = RUN_ROOT / 'yellow_lego_selected.ply'
run([sys.executable, REPO / 'semantic_query.py', '--model', MODEL,
     '--text', 'yellow Lego bulldozer', '--iteration', str(RGB_ITERATIONS),
     '--threshold', '-1.0', '--top_k', '500', '--output', SELECTION,
     '--json', QUERY_JSON, '--export_selected', SELECTED_PLY, '--device', 'cuda'], cwd=REPO)
if EDITED_MODEL.exists():
    shutil.rmtree(EDITED_MODEL)
run([sys.executable, REPO / 'semantic_edit.py', '--model', MODEL,
     '--selection', SELECTION, '--output_model', EDITED_MODEL, '--action', 'remove'], cwd=REPO)

from plyfile import PlyData
original_count = len(PlyData.read(RGB_PLY)['vertex'].data)
selected_count = len(np.load(SELECTION)['indices'])
edited_ply = EDITED_MODEL / 'point_cloud' / f'iteration_{RGB_ITERATIONS}' / 'point_cloud.ply'
edited_count = len(PlyData.read(edited_ply)['vertex'].data)
assert selected_count > 0
assert edited_count == original_count - selected_count
assert RGB_PLY.is_file(), '源模型不应被删除操作修改'
result = json.loads(QUERY_JSON.read_text())
print(json.dumps(result, ensure_ascii=False, indent=2))
print(f'删除验证通过：{original_count} - {selected_count} = {edited_count}')


## 7. 可旋转、俯仰和缩放的实时 3DGS 渲染器

这一节不是点云，也不是视频。为了让展示比 300 次冒烟训练更清晰，先在 T4 上额外训练一个 3,000 次迭代的展示模型，然后提供水平旋转、俯仰和缩放滑块；每次操作都会使用 Gaussian rasterizer 从新相机位姿实时渲染。

In [ ]:
DISPLAY_ITERATIONS = 3000
DISPLAY_MODEL = RUN_ROOT / 'model_render_3000'
DISPLAY_PLY = DISPLAY_MODEL / 'point_cloud' / f'iteration_{DISPLAY_ITERATIONS}' / 'point_cloud.ply'
if not DISPLAY_PLY.is_file():
    run([sys.executable, REPO / 'train.py',
         '-s', SCENE, '-m', DISPLAY_MODEL, '--eval', '--white_background',
         '--iterations', str(DISPLAY_ITERATIONS),
         '--save_iterations', str(DISPLAY_ITERATIONS),
         '--test_iterations', str(DISPLAY_ITERATIONS),
         '--importance_mask_dir', SCENE / 'importance_masks',
         '--foreground_weight', '4.0', '--background_weight', '0.25',
         '--densify_from_iter', '500', '--densify_until_iter', '2500',
         '--densification_interval', '100', '--opacity_reset_interval', '3000',
         '--disable_viewer', '--quiet'], cwd=REPO)
assert DISPLAY_PLY.is_file(), DISPLAY_PLY
print('✓ 交互展示模型：', DISPLAY_PLY)


In [ ]:
from interactive_renderer import show_interactive_renderer

INTERACTIVE_VIEWER = show_interactive_renderer(
    DISPLAY_MODEL, iteration=DISPLAY_ITERATIONS, width=512, up_axis='z'
)


In [ ]:
archive = shutil.make_archive('/content/semantic_3dgs_t4_smoke_results', 'zip', RUN_ROOT)
print('✅ 全链路 T4 冒烟测试通过')
print('结果压缩包:', archive)
print('需要下载时运行：from google.colab import files; files.download(archive)')
